# Basic Preprocessing

## Import packages, indicate the /src location, retrieve the data, and prep the corpus for segmentation

In [ ]:
# ! pip install sentence_transformers

In [1]:
# === Import
import pandas as pd
import re
import sys
import json
from pathlib import Path
import nltk

# === Download NLTK resources if missing ===
try: nltk.data.find("tokenizers/punkt")
except LookupError: nltk.download("punkt")
try: nltk.data.find("tokenizers/punkt_tab")
except LookupError:
    try: nltk.download("punkt_tab")
    except Exception: pass

# === Define the path to the auxiliary modules ===
def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()

    for candidate in (start, *start.parents):
        if (candidate / ".git").exists() or (candidate / "pyproject.toml").exists():
            return candidate

    raise RuntimeError("Could not find project root.")

ROOT = find_repo_root()
SRC = ROOT / "src"

if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

In [2]:
# === Define the path to the data and the pattern for retrieval ==
HOME = Path.home()
DATA_DIR = (HOME / "My Drive" / "VectorData" / "projects" / "identifying_depression_with_rst" / "data").resolve(strict=True)

# === Pattern ===
data_files_pattern = r"K.+\.csv"

# === Retrieve the data
find_files = DATA_DIR / "raw"

data = []

for item in find_files.iterdir():
   if item.is_file() and re.search(data_files_pattern, item.name):
      data.append((item.stem.lower(), pd.read_csv(item)))


## Coverting the Data into Corpora

In [3]:
# The structure of the corpora is as follows:
# {"name_of_corpus": ['document_1', 'document_2'], ...}

processed_corpora = {}
for name, corpus in data:
    processed_corpora.setdefault(name, []).extend(corpus["text"].to_list())

# The database of diagnonses labels mimics the corpora structure
# e.g. {"name_of_corpus": ["diagnosis_1", "diagnonsis_2", "diagnosis_1"]}

diagnoses = {}
for name, corpus in data:
    diagnoses.setdefault(name, []).extend(corpus["group"].to_list())

# Saving the corpus/corpora for downstream processing (with an RST parser)

In [4]:
CORPORA = processed_corpora # {"coprus": [doc_1, doc_2, ....]}

In [5]:
save_files_path = DATA_DIR / "interim"
processed_data_file = save_files_path / "preprocesssed_corpora.json"

with open(processed_data_file, "w") as file:
    json.dump(CORPORA, file, indent=4, ensure_ascii=False)

In [7]:
# Save all the diagnonses labels into a separate file that mimics the corpora structure
# e.g. {"ked": ["diagnosis_1", "diagnonsis_2", "diagnosis_1"]}

save_files_path = DATA_DIR / "interim"
diagnoses_data = save_files_path / "all_diagnoses.json"

with open(diagnoses_data, "w") as file:
    json.dump(diagnoses, file, indent=4, ensure_ascii=False)